# Notebook 12 — Prescriptive Analytics

**Tier:** 4 — Prescriptive (What should we do?)  
**Purpose:** Convert insights from Tiers 1–3 into actionable recommendations with quantified scenario estimates.  
**Input:** `outputs/feature_store.parquet`, outputs from prior notebooks  
**Output:** `outputs/reports/prescriptions.csv`, charts  

> **Important:** Scenario estimates are directional upper bounds only.  
> CAC, LTV, ROAS, driver payout, commission, operating cost, and contribution margin
> are **unavailable** — all prescriptions are revenue-side only (Gross Booking Value).
> ROI cannot be quantified without cost data.

In [20]:
import os
os.chdir(r"D:\projects\Uber_Data_India_Analytics")
import sys, warnings, pathlib
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
pathlib.Path('outputs/plots').mkdir(parents=True, exist_ok=True)
pathlib.Path('outputs/reports').mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

df = pd.read_parquet('outputs/feature_store.parquet')
completed = df[df['is_completed']].copy()
slot_order = ['Early Morning','Morning','Afternoon','Evening','Night','Late Night']
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print(f'Loaded feature store: {df.shape}  |  completed: {len(completed):,}')

Loaded feature store: (150000, 46)  |  completed: 93,000


## 12.1 Supply Optimisation Prescriptions

In [21]:
# Identify top (zone, time_slot) cells with highest No-Driver-Found rate
ndf = df.groupby(['pickup_zone','time_slot'], observed=True).agg(
    total=('Booking ID','count'),
    ndf_count=('Booking Status', lambda s: (s=='No Driver Found').sum()),
).reset_index()
ndf['ndf_rate_pct'] = (ndf['ndf_count'] / ndf['total'] * 100).round(1)
ndf = ndf[ndf['total'] >= 50].sort_values('ndf_rate_pct', ascending=False).head(10)
print('Top 10 (Zone, Time Slot) cells by No-Driver-Found rate (min 50 bookings):')
print(ndf.to_string(index=False))

Top 10 (Zone, Time Slot) cells by No-Driver-Found rate (min 50 bookings):
pickup_zone     time_slot  total  ndf_count  ndf_rate_pct
  Faridabad       Evening    259         24           9.3
  Faridabad Early Morning     57          5           8.8
  Outer NCR Early Morning    362         32           8.8
  Faridabad     Afternoon    154         13           8.4
  Outer NCR    Late Night     97          8           8.2
  Outer NCR       Evening   1725        139           8.1
    Gurgaon    Late Night    323         26           8.0
Delhi North Early Morning   1256         99           7.9
      Noida Early Morning    525         41           7.8
Delhi South Early Morning   1415        110           7.8


In [22]:
# Scenario: reduce NDF rate in top cells by 20% → estimated additional rides × median fare
median_fare = completed['Booking Value'].median()
avg_fare = completed['Booking Value'].mean()

ndf['scenario_extra_rides'] = (ndf['ndf_count'] * 0.20).round(0).astype(int)
ndf['scenario_gbv_uplift'] = (ndf['scenario_extra_rides'] * median_fare).round(0)
print('Scenario: 20% NDF reduction in top cells (using median fare ₹{:.0f}):'.format(median_fare))
print(ndf[['pickup_zone','time_slot','ndf_count','scenario_extra_rides','scenario_gbv_uplift']].to_string(index=False))
print(f'\nTotal scenario GBV uplift: ₹{ndf["scenario_gbv_uplift"].sum():,.0f}')
print()
print('⚠ SCENARIO ESTIMATE ONLY — assumes recovered rides complete at median fare.')
print('  Actual uplift depends on whether drivers are available and rides complete.')
print('  Net revenue impact cannot be calculated (no driver payout or cost data).')

Scenario: 20% NDF reduction in top cells (using median fare ₹414):
pickup_zone     time_slot  ndf_count  scenario_extra_rides  scenario_gbv_uplift
  Faridabad       Evening         24                     5               2070.0
  Faridabad Early Morning          5                     1                414.0
  Outer NCR Early Morning         32                     6               2484.0
  Faridabad     Afternoon         13                     3               1242.0
  Outer NCR    Late Night          8                     2                828.0
  Outer NCR       Evening        139                    28              11592.0
    Gurgaon    Late Night         26                     5               2070.0
Delhi North Early Morning         99                    20               8280.0
      Noida Early Morning         41                     8               3312.0
Delhi South Early Morning        110                    22               9108.0

Total scenario GBV uplift: ₹41,400

⚠ SCENARIO ESTIM

## 12.2 Cancellation Reduction Prescriptions

In [23]:
# Customer cancellation: VTAT effect
cust_canc = df[df['Booking Status'] == 'Cancelled by Customer'].copy()
all_dispatched = df[df['Avg VTAT'].notna()].copy()
avg_vtat_canc = cust_canc['Avg VTAT'].dropna().mean()
avg_vtat_all  = all_dispatched['Avg VTAT'].mean()

print(f'Avg VTAT — customer-cancelled: {avg_vtat_canc:.2f} min')
print(f'Avg VTAT — all dispatched    : {avg_vtat_all:.2f} min')

# Scenario: reduce customer cancellation rate by 10%
n_cust_canc = len(cust_canc)
scenario_saved = int(n_cust_canc * 0.10)
scenario_gbv = scenario_saved * median_fare
print(f'\nScenario: 10% reduction in customer cancellations → {scenario_saved:,} extra rides')
print(f'Estimated GBV uplift: ₹{scenario_gbv:,.0f}  (at median fare — upper-bound estimate)')
print()
print('Recommended interventions:')
print('  1. Alert driver if VTAT exceeds threshold (~12 min) to prompt movement toward pickup.')
print('  2. Monitor "driver not moving" cases (reason category) and surface in driver app.')
print('  3. Notify customer with ETA update if VTAT exceeds 10 min to manage expectations.')

Avg VTAT — customer-cancelled: 12.51 min
Avg VTAT — all dispatched    : 8.46 min

Scenario: 10% reduction in customer cancellations → 1,050 extra rides
Estimated GBV uplift: ₹434,700  (at median fare — upper-bound estimate)

Recommended interventions:
  1. Alert driver if VTAT exceeds threshold (~12 min) to prompt movement toward pickup.
  2. Monitor "driver not moving" cases (reason category) and surface in driver app.
  3. Notify customer with ETA update if VTAT exceeds 10 min to manage expectations.


## 12.3 Customer Retention Prescriptions

In [24]:
from src.cohort import build_cohort_retention, build_cohort_retention_rate, avg_retention_curve

count_matrix = build_cohort_retention(df)
rate_matrix  = build_cohort_retention_rate(count_matrix)
curve        = avg_retention_curve(rate_matrix)

if 0 in curve.index and 1 in curve.index:
    m1_ret = curve[1]
    drop = curve[0] - m1_ret
    print(f'Month-0 retention: {curve[0]:.1f}%')
    print(f'Month-1 retention: {m1_ret:.1f}%')
    print(f'Month-0 → Month-1 drop: {drop:.1f} pp  ← primary churn event')
    print()
    print('Prescription: Re-engagement notification at Day 7 after first ride.')
    print('Target: increase Month-1 retention from {:.1f}% toward benchmark.'.format(m1_ret))
    print()
    print('NOTE: Revenue impact of retention actions cannot be quantified without CAC data.')
    print('      Prescriptions are directional — ROI requires cost-side data.')

Month-0 retention: 100.0%
Month-1 retention: 0.1%
Month-0 → Month-1 drop: 99.9 pp  ← primary churn event

Prescription: Re-engagement notification at Day 7 after first ride.
Target: increase Month-1 retention from 0.1% toward benchmark.

NOTE: Revenue impact of retention actions cannot be quantified without CAC data.
      Prescriptions are directional — ROI requires cost-side data.


In [25]:
from src.segmentation import build_rfm
rfm = build_rfm(df)
seg_actions = {
    'Champion'          : 'Loyalty tier recognition; priority support; early access to new features.',
    'Loyal'             : 'Reward milestone (e.g. 10th ride discount); referral incentive.',
    'Potential Loyalist': 'Second-ride incentive within 14 days of first booking.',
    'New Customer'      : 'Onboarding message + second-ride discount at Day 7.',
    'At Risk'           : 'Re-engagement offer; personalised push notification.',
    'Hibernating'       : 'Win-back campaign with time-limited discount.',
    'Lost'              : 'Minimal spend; opt-out from paid campaigns until re-activated.',
}
seg_counts = rfm['rfm_segment'].value_counts()
print('RFM Segment Actions:')
for seg, action in seg_actions.items():
    count = seg_counts.get(seg, 0)
    print(f'  {seg:22s} ({count:,} customers): {action}')
print()
print('NOTE: CAC and LTV are unavailable. All prescriptions are directional.')

RFM Segment Actions:
  Champion               (2,129 customers): Loyalty tier recognition; priority support; early access to new features.
  Loyal                  (35,486 customers): Reward milestone (e.g. 10th ride discount); referral incentive.
  Potential Loyalist     (37,182 customers): Second-ride incentive within 14 days of first booking.
  New Customer           (0 customers): Onboarding message + second-ride discount at Day 7.
  At Risk                (27,711 customers): Re-engagement offer; personalised push notification.
  Hibernating            (9,306 customers): Win-back campaign with time-limited discount.
  Lost                   (36,974 customers): Minimal spend; opt-out from paid campaigns until re-activated.

NOTE: CAC and LTV are unavailable. All prescriptions are directional.


## 12.4 Peak Hour Demand Management

In [26]:
# Demand-supply gap index by hour: bookings / completed rides
hourly_gap = df.groupby('hour').agg(
    bookings=('Booking ID','count'),
    completed=('is_completed','sum'),
).assign(gap_index=lambda x: (x['bookings'] / x['completed'].replace(0,1)).round(2))
hourly_gap = hourly_gap.sort_values('gap_index', ascending=False)
print('Hours with highest demand-supply gap (bookings per completed ride):')
print(hourly_gap.head(6).to_string())
print()
print('Prescription: Consider dynamic pricing (surge) during peak gap hours.')
print('Prescription: Driver scheduling incentives for hours with gap_index > 1.5.')

Hours with highest demand-supply gap (bookings per completed ride):
      bookings  completed  gap_index
hour                                
1         1360        828       1.64
7         5450       3346       1.63
19       11047       6798       1.63
14        7031       4310       1.63
18       12397       7617       1.63
9         8234       5084       1.62

Prescription: Consider dynamic pricing (surge) during peak gap hours.
Prescription: Driver scheduling incentives for hours with gap_index > 1.5.


## 12.5 Payment Mix Optimisation

In [27]:
payment_dist = completed['Payment Method'].value_counts()
cash_pct = payment_dist.get('Cash', 0) / payment_dist.sum() * 100
upi_pct  = payment_dist.get('UPI',  0) / payment_dist.sum() * 100
print(f'UPI  : {upi_pct:.1f}% of completed rides')
print(f'Cash : {cash_pct:.1f}% of completed rides')
print()
print('Prescriptions:')
print('  1. UPI cashback or loyalty points to consolidate UPI dominance.')
print('  2. In-app nudge + incentive on cash rides to migrate to digital payment.')
print('  3. Focus cash-to-digital migration on top cash-heavy routes (see Notebook 07).')

UPI  : 45.0% of completed rides
Cash : 24.9% of completed rides

Prescriptions:
  1. UPI cashback or loyalty points to consolidate UPI dominance.
  2. In-app nudge + incentive on cash rides to migrate to digital payment.
  3. Focus cash-to-digital migration on top cash-heavy routes (see Notebook 07).


## 12.6 Prescription Summary

In [28]:
prescriptions = [
    {'area': 'Supply Optimisation',     'action': 'Driver incentives in top NDF (zone, slot) cells',
     'scenario_gbv_uplift_inr': int(ndf['scenario_gbv_uplift'].sum()),
     'confidence': 'Upper-bound estimate', 'requires_cost_data': False},
    {'area': 'Cancellation Reduction',  'action': '10% customer cancellation reduction (VTAT alerts)',
     'scenario_gbv_uplift_inr': int(scenario_gbv),
     'confidence': 'Upper-bound estimate', 'requires_cost_data': False},
    {'area': 'Retention',               'action': 'Day-7 re-engagement for all new customers',
     'scenario_gbv_uplift_inr': None,
     'confidence': 'Directional only — CAC/cost data needed for ROI', 'requires_cost_data': True},
    {'area': 'Peak Demand',             'action': 'Surge pricing / driver scheduling for high-gap hours',
     'scenario_gbv_uplift_inr': None,
     'confidence': 'Directional only — pricing model needed', 'requires_cost_data': False},
    {'area': 'Payment Mix',             'action': 'UPI cashback; cash-to-digital migration',
     'scenario_gbv_uplift_inr': None,
     'confidence': 'Directional only — incentive cost data needed', 'requires_cost_data': True},
]
presc_df = pd.DataFrame(prescriptions)
print(presc_df.to_string(index=False))
presc_df.to_csv('outputs/reports/prescriptions.csv', index=False)
print('\nSaved: outputs/reports/prescriptions.csv')

                  area                                               action  scenario_gbv_uplift_inr                                      confidence  requires_cost_data
   Supply Optimisation      Driver incentives in top NDF (zone, slot) cells                  41400.0                            Upper-bound estimate               False
Cancellation Reduction    10% customer cancellation reduction (VTAT alerts)                 434700.0                            Upper-bound estimate               False
             Retention            Day-7 re-engagement for all new customers                      NaN Directional only — CAC/cost data needed for ROI                True
           Peak Demand Surge pricing / driver scheduling for high-gap hours                      NaN         Directional only — pricing model needed               False
           Payment Mix              UPI cashback; cash-to-digital migration                      NaN   Directional only — incentive cost data needed       

## Unavailable Metrics (documented)

| Metric | Why unavailable |
|---|---|
| CAC | No acquisition channel or marketing spend field |
| LTV | No cost-side data; cannot compute net margin |
| ROAS | No ad spend data |
| Driver payout | No payout rate or amount field |
| Platform commission | No commission rate field |
| Operating cost | No cost data of any kind |
| Contribution margin | Requires revenue minus variable cost; cost absent |

These metrics would require additional data from financial and CRM systems not included in this dataset.